# Pandas Data Engineering Interview Practice — 80 Questions + Executable Solutions

8 categories × 10 questions. Run the setup cells first, then write your answer in the **Python code cell directly below each question**. Each prompt shows the exact datasets and columns, with compact line breaks.

The datasets are bundled in `data/`; no PostgreSQL server or password is required. The solutions notebook uses the same data and setup.


> **Realistic Dataset v2:** Shared across Python, Pandas, and PySpark. Existing columns retained. Synthetic data includes intentional dirty records and malformed JSON for interview exercises.


## Environment setup

In [ ]:
# Install the pinned requirements.txt in the practice virtual environment first.
# Select that environment as the notebook kernel; see README.md for setup/preflight.
# Package installation is intentionally not performed by this notebook.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Keep this notebook and the data/ folder together.
DATA_DIR = Path('data')
if not DATA_DIR.exists():
    DATA_DIR = Path('pandas_de_practice/data')
assert DATA_DIR.exists(), 'Cannot find data/. Open the extracted pandas_de_practice folder in VS Code.'

customers = pd.read_csv(DATA_DIR / 'customers.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
transactions = pd.read_csv(DATA_DIR / 'transactions.csv')
pipeline_logs = pd.read_csv(DATA_DIR / 'pipeline_logs.csv')
cdc_changes = pd.read_csv(DATA_DIR / 'cdc_changes.csv')
events = pd.read_json(DATA_DIR / 'events.jsonl', lines=True)
events_duplicates = pd.read_json(DATA_DIR / 'events_duplicates.jsonl', lines=True)
customers_dirty = pd.read_json(DATA_DIR / 'customers_dirty.json')

print('Loaded:', {k:len(globals()[k]) for k in ['customers','orders','order_items','products','transactions','pipeline_logs','cdc_changes','events','events_duplicates','customers_dirty']})


## 1. DataFrame Fundamentals


### Q01 — Preview records

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Return the first 10 rows of the `orders` DataFrame in its existing row order. Retain all original columns and the corresponding index values; do not alter the source DataFrame.

**Requirements:**
- Select by current row position, not by the numeric values of the index.
- Return all available rows if `orders` contains fewer than ten.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame containing up to ten complete order rows, in their existing order with their original index values. An empty input produces an empty DataFrame with the original columns.

**Solution:**


In [ ]:
result = orders.head(10)

display(result)


### Q02 — Column projection

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Create a DataFrame from `orders` containing only `order_id`, `customer_id`, and `amount`, in that column order. Retain every row in its existing order with its original index. Do not change the source DataFrame.

**Requirements:**
- Projection must not filter or deduplicate records.
- Retain the selected columns' values and dtypes without resetting the index.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with exactly `order_id`, `customer_id`, and `amount`, in that order. It has the same rows, row order, and index as `orders`.

**Solution:**


In [ ]:
result = orders[['order_id','customer_id','amount']]

display(result)


### Q03 — Filter completed

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Return all rows of `orders` whose `status` is exactly `completed`. Preserve every original column, the order of the matching rows, and their index values. The original DataFrame should remain unchanged.

**Requirements:**
- Use an exact status comparison without trimming or case normalization.
- Do not reset the index or remove duplicate qualifying rows.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame of completed orders with all original columns. Matching rows retain their input order and index; no matches produces an empty DataFrame with the same columns.

**Solution:**


In [ ]:
result = orders.loc[orders['status'].eq('completed')].copy()

display(result)


### Q04 — High-value orders

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Identify orders in `orders` with `amount` of at least 300, including orders exactly at the threshold. Return the complete qualifying rows sorted by amount descending. Keep their original index values and do not modify the input.

**Requirements:**
- Missing amounts do not qualify for the numeric threshold.
- Sort by amount only; no additional tie-breaker is required.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame containing complete order rows with `amount` at least 300, sorted by amount descending. Each selected row retains its original index.

**Solution:**


In [ ]:
result = orders.loc[orders['amount'].ge(300)].sort_values('amount',ascending=False)

display(result)


### Q05 — Rename columns

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Create a version of `orders` in which the `amount` column is named `order_amount`. Preserve its values, all other columns, and every row and index value. Leave the original DataFrame and its column names unchanged.

**Requirements:**
- Change only the column label, not the amount values or dtype.
- Rename the column in the result without also retaining a duplicate `amount` column.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with `order_id`, `customer_id`, `order_date`, `status`, and `order_amount`. Rows, index values, and data are identical to `orders`; only the `amount` label changes.

**Solution:**


In [ ]:
result = orders.rename(columns={'amount':'order_amount'})

display(result)


### Q06 — Create derived field

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Add an `amount_cad` column to a new result DataFrame by multiplying each `orders.amount` value by the illustrative conversion rate 1.35. Retain every original row and column, including the unchanged `amount` values, and preserve the input index.

**Requirements:**
- Apply the fixed illustrative rate to each row independently; no currency lookup is needed.
- Keep the input DataFrame unchanged and retain missing amounts as missing derived values.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with every original `orders` column plus `amount_cad`, equal to the corresponding amount multiplied by 1.35. Row count, row order, and index match the input.

**Solution:**


In [ ]:
result = orders.assign(amount_cad=orders['amount']*1.35)

display(result)


### Q07 — Distinct statuses

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Return a one-column DataFrame with a `status` column containing the distinct non-missing status values in `orders`. Sort the values in ascending order and include each status once, regardless of how many orders use it.

**Requirements:**
- Exclude missing statuses before forming the distinct result.
- Keep status text as stored; do not merge differently cased or spaced values.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with one `status` column and one row per distinct non-missing status, sorted ascending. No counts or original order columns are included.

**Solution:**


In [ ]:
result = pd.DataFrame({'status': sorted(orders['status'].dropna().unique())})

display(result)


### Q08 — Missing-value profile

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Profile missing data in `customers`. Return a DataFrame with one row per original column and two fields: `column`, containing its name, and `missing_count`, containing the number of missing values. Include columns with zero missing values and retain the input column order.

**Requirements:**
- Count missing cells separately for each `customers` column without dropping records before counting.
- Assess missing values as loaded; leave the customer data unchanged rather than cleaning or validating its text fields.

**Available datasets and columns:**
**`customers:`** customer_id, name, email, segment, country, signup_date

**Expected output:** A Pandas DataFrame with columns `column` and `missing_count`. Include one row for every original `customers` column, in the same column order, with its name and integer count of missing values. Columns without missing values have a count of zero.

**Solution:**


In [ ]:
result = customers.isna().sum().rename('missing_count').reset_index().rename(columns={'index':'column'})

display(result)


### Q09 — DataFrame metadata

**Interview scenario:** You are building a product catalog data mart. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Inspect the columns of `products` and return a DataFrame with `column` and `dtype`. Include one row for every input column in its existing order, with each dtype represented as text. This task reports metadata without changing any product values or types.

**Requirements:**
- Report the dtypes currently loaded, not proposed or inferred replacement types.
- Include every product column even when it contains no missing values.

**Available datasets and columns:**
**`products:`** product_id, sku, category, price

**Expected output:** A Pandas DataFrame with `column` and `dtype`, one row per `products` column in original column order. Both fields contain text, and the product data is unchanged.

**Solution:**


In [ ]:
result = products.dtypes.astype(str).rename('dtype').reset_index().rename(columns={'index':'column'})

display(result)


### Q10 — Conditional assignment

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Add `value_band` to a copy of `orders`. Label an order `high` when `amount` is at least 250 and `standard` otherwise, including when the amount is missing. Preserve all rows, original columns, and index values.

**Requirements:**
- Use the exact labels `high` and `standard`.
- Missing amounts take the `standard` label; do not remove those orders.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with all original order columns plus `value_band`, one label per order. Amounts of 250 or more are `high`; all other rows are `standard`. The original row order and index are retained.

**Solution:**


In [ ]:
result = orders.assign(value_band=np.where(orders['amount'].ge(250),'high','standard'))

display(result)


## 2. Filtering, Sorting & Indexing


### Q11 — Multi-condition filter

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Return complete rows from `orders` where `status` is exactly `completed` and `amount` is at least 200. Both conditions must hold for the same order. Preserve the qualifying rows' existing order and index values.

**Requirements:**
- Include the amount boundary of 200 and match `completed` exactly.
- Do not qualify an order that satisfies only one condition or has a missing amount.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame containing complete completed-order rows with `amount` at least 200. It preserves their original order and index, with no additional summary columns.

**Solution:**


In [ ]:
result = orders.loc[orders['status'].eq('completed') & orders['amount'].ge(200)]

display(result)


### Q12 — Membership filter

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Filter `customers` to records whose `country` is either `CA` or `US`. Match these stored codes exactly and retain all original columns, row order, and index values for the matching customers.

**Requirements:**
- Do not translate country names or normalize differently formatted codes.
- Keep every qualifying customer row, including repeated records.

**Available datasets and columns:**
**`customers:`** customer_id, name, email, segment, country, signup_date

**Expected output:** A Pandas DataFrame containing only `CA` and `US` customer records, with all original columns and the selected rows' original order and index.

**Solution:**


In [ ]:
result = customers.loc[customers['country'].isin(['CA','US'])]

display(result)


### Q13 — Exclude cancelled

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Return rows from `orders` whose `status` is not `cancelled`, using the values as loaded in the provided DataFrame. Keep the original columns and the order and index values of qualifying rows. Leave the source data unchanged.

**Requirements:**
- Compare status values as loaded without filling missing values or normalizing text.
- Do not modify `orders` or reset the indexes of retained rows.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame containing complete order rows satisfying the stated not-cancelled comparison. Selected rows retain their original order, columns, and index values.

**Solution:**


In [ ]:
result = orders.loc[orders['status'].ne('cancelled')]

display(result)


### Q14 — Top 5 amounts

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Return the five rows of `orders` with the largest `amount` values, arranged from largest to smallest. Include all original columns and retain the selected rows' indexes. The result should contain no more than five orders, rather than five distinct amount levels. For equal amounts, retain the original input order, including when a tie crosses the five-row cutoff.

**Requirements:**
- Apply the limit to records, not distinct amount levels.
- Retain input order among equal amounts, including at the cutoff, and leave `orders` unchanged.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame of up to five complete orders ranked by amount descending, with original indexes. Tied amounts keep their input order; fewer than five input rows produces a correspondingly smaller result.

**Solution:**


In [ ]:
result = orders.nlargest(5,'amount')

display(result)


### Q15 — Sort by two columns

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Sort all records in `customers` by `country` ascending and then by `signup_date` from earliest to latest within each country. Retain all columns and carry the original index values with their rows. Return a sorted result without modifying the input.

**Requirements:**
- Keep signup dates and other field values in their existing representation.
- Apply no additional customer-ID tie-breaker; leave the source DataFrame unchanged.

**Available datasets and columns:**
**`customers:`** customer_id, name, email, segment, country, signup_date

**Expected output:** A Pandas DataFrame containing every customer row and column, ordered by country ascending and then signup date ascending. Original index values travel with their rows.

**Solution:**


In [ ]:
result = customers.sort_values(['country','signup_date'])

display(result)


### Q16 — Index by key

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Create a DataFrame from `customers` whose index is `customer_id`. Keep `customer_id` as a regular column as well, and retain every other column and row in its existing order. Leave the input DataFrame's index unchanged.

**Requirements:**
- Use customer-ID values as index labels without sorting or deduplicating rows.
- Retain `customer_id` as both an index named `customer_id` and a regular column.

**Available datasets and columns:**
**`customers:`** customer_id, name, email, segment, country, signup_date

**Expected output:** A Pandas DataFrame containing all customer columns and rows in their existing order, indexed by `customer_id`. The source DataFrame keeps its original index.

**Solution:**


In [ ]:
result = customers.set_index('customer_id',drop=False)

display(result)


### Q17 — Case-insensitive match

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Find customers in `customers` whose `segment` equals `Enterprise`, ignoring letter case. Return their complete records with the original row order and index values. A substring match, such as a longer segment containing that word, is not sufficient.

**Requirements:**
- Ignore case for matching only; do not rewrite segment values in the output.
- Missing segments and longer strings containing the word do not qualify.

**Available datasets and columns:**
**`customers:`** customer_id, name, email, segment, country, signup_date

**Expected output:** A Pandas DataFrame of complete customer rows whose segment is a case-insensitive exact match for `Enterprise`, retaining original values, row order, and index.

**Solution:**


In [ ]:
result = customers.loc[customers['segment'].str.casefold().eq('enterprise')]

display(result)


### Q18 — String contains

**Interview scenario:** You are building an ETL observability system. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Return complete `pipeline_logs` rows whose `message` contains either `error` or `fail`, ignoring letter case. Match these character sequences anywhere in the message and exclude missing messages. Preserve all log columns and the qualifying rows' order and indexes.

**Requirements:**
- Search message text rather than the log's `level` field.
- A row matching both character sequences appears once; missing messages are excluded.

**Available datasets and columns:**
**`pipeline_logs:`** log_id, pipeline, level, duration_ms, timestamp, message

**Expected output:** A Pandas DataFrame of complete matching log rows in their original order and with original indexes. The result retains the original message text and all other log columns.

**Solution:**


In [ ]:
result = pipeline_logs.loc[pipeline_logs['message'].str.contains('error|fail',case=False,na=False,regex=True)]

display(result)


### Q19 — Range filter

**Interview scenario:** You are building a multi-currency payment ingestion service. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Return transaction records from `transactions` whose `amount` is between 100 and 500, including both endpoints. Preserve all original columns and the matching rows' order and index values. Missing amounts do not qualify.

**Requirements:**
- Include amounts equal to either endpoint and exclude missing amounts.
- Keep repeated qualifying transactions and leave the source DataFrame unchanged.

**Available datasets and columns:**
**`transactions:`** transaction_id, customer_id, amount, currency, transaction_ts, type

**Expected output:** A Pandas DataFrame of complete transaction rows with amounts from 100 through 500 inclusive, preserving their original row order and index.

**Solution:**


In [ ]:
result = transactions.loc[transactions['amount'].between(100,500)]

display(result)


### Q20 — Stable sort

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Order all rows of `orders` by `amount` descending and then `order_id` ascending. Use a stable sort so records equal on both keys retain their original relative order. Preserve all columns and carry each row's original index into the result.

**Requirements:**
- Apply the stable ordering required by the Task without adding another tie-breaker.
- Retain duplicate rows and leave `orders` unchanged.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame containing all order rows and columns, ordered by amount descending and order ID ascending. Rows equal on both keys keep their input order, and original indexes are preserved.

**Solution:**


In [ ]:
result = orders.sort_values(['amount','order_id'],ascending=[False,True],kind='stable')

display(result)


## 3. Grouping & Aggregations


### Q21 — Status counts

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Produce one row per distinct `status` value in the `orders` DataFrame, including missing status values. Return the status and its number of order rows as `order_count`. Do not modify the original DataFrame.

**Requirements:**
- Count rows even when their amounts or other fields are missing.
- Retain a separate missing-status group and do not change `orders`.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with `status` and integer `order_count`, one row per observed status including missing status. The counts together equal the number of input orders.

**Solution:**


In [ ]:
result = orders.groupby('status',dropna=False).size().reset_index(name='order_count')

display(result)


### Q22 — Revenue by status

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Calculate total order revenue for each non-missing `status` in `orders`. Return one row per status with `status` and `revenue`, where revenue is the sum of its `amount` values. The result is a grouped summary, not an annotation of individual orders.

**Requirements:**
- Exclude missing status keys from the summary.
- Sum available amounts without rounding individual orders; leave the input unchanged.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with `status` and `revenue`, one row per non-missing status. `revenue` is the grouped sum of order amounts, not a count or an average.

**Solution:**


In [ ]:
result = orders.groupby('status',as_index=False).agg(revenue=('amount','sum'))

display(result)


### Q23 — Customer spend

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Summarize `orders` by non-missing `customer_id`, including orders of every status. Return one row per customer with `customer_id` and `total_spend`, calculated from `amount`. Customer IDs with no order records need not be added to the result.

**Requirements:**
- Do not restrict the calculation to completed orders or require a customer-table match.
- Exclude missing customer IDs and retain the input data unchanged.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with `customer_id` and `total_spend`, one row per non-missing customer ID observed in `orders`. Each total includes amounts across all of that customer's order statuses.

**Solution:**


In [ ]:
result = orders.groupby('customer_id',as_index=False).agg(total_spend=('amount','sum'))

display(result)


### Q24 — Order KPIs

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Produce order statistics for each non-missing `status` in `orders`. Return `status`, `order_count`, `avg_amount`, `min_amount`, and `max_amount`. Count all order rows in each status; calculate the amount statistics from non-missing amounts. Do not return individual order records.

**Requirements:**
- An order with a missing amount still contributes to `order_count`.
- Exclude missing status keys and do not replace missing amounts with zero before calculating statistics.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with `status`, `order_count`, `avg_amount`, `min_amount`, and `max_amount`, one row per non-missing status. Amount statistics use available amounts; groups without any have missing amount statistics.

**Solution:**


In [ ]:
result = orders.groupby('status',as_index=False).agg(order_count=('order_id','size'),avg_amount=('amount','mean'),min_amount=('amount','min'),max_amount=('amount','max'))

display(result)


### Q25 — Segment distribution

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Count customer records for each combination of `segment` and `country` in `customers`. Return a DataFrame with `segment`, `country`, and `customers` as the row count. Exclude records missing either grouping value and include only combinations present in the data.

**Requirements:**
- Count customer records, not distinct customer IDs.
- Omit records missing either grouping field and do not create unobserved segment-country combinations.

**Available datasets and columns:**
**`customers:`** customer_id, name, email, segment, country, signup_date

**Expected output:** A Pandas DataFrame with `segment`, `country`, and `customers`. Each row represents one observed non-missing segment-country pair, and `customers` is its integer record count.

**Solution:**


In [ ]:
result = customers.groupby(['segment','country'],as_index=False).size().rename(columns={'size':'customers'})

display(result)


### Q26 — Completed revenue

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Calculate revenue per customer from rows in `orders` whose `status` is exactly `completed`. Return `customer_id` and `completed_revenue`, summing `amount` for each non-missing customer ID. Include only customers represented by qualifying completed orders.

**Requirements:**
- Match `completed` exactly and exclude missing customer IDs.
- Do not add customers with no qualifying orders or mix other statuses into their totals.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with `customer_id` and `completed_revenue`, one row per customer represented by completed orders. The revenue is the sum of that customer's qualifying amounts.

**Solution:**


In [ ]:
result = orders.loc[orders['status'].eq('completed')].groupby('customer_id',as_index=False)['amount'].sum().rename(columns={'amount':'completed_revenue'})

display(result)


### Q27 — Product units

**Interview scenario:** You are building an e-commerce order-line fact pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Summarize `order_items` by `product_id`. Return one row per product with `product_id` and `units_sold`, where units sold is the total `quantity` across its order items. Count quantities rather than item records.

**Requirements:**
- Include every order-item occurrence in the quantity totals.
- Do not add products absent from `order_items` or use prices in this units calculation.

**Available datasets and columns:**
**`order_items:`** item_id, order_id, product_id, quantity, unit_price

**Expected output:** A Pandas DataFrame with `product_id` and `units_sold`, one row per represented non-missing product ID. `units_sold` contains summed item quantities.

**Solution:**


In [ ]:
result = order_items.groupby('product_id',as_index=False).agg(units_sold=('quantity','sum'))

display(result)


### Q28 — Line-item sales

**Interview scenario:** You are building an e-commerce order-line fact pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Calculate product sales from `order_items`, using each item's `quantity` multiplied by its `unit_price`. Return one row per `product_id` with the summed value as `sales`. Use the sale-time price in the item record, not a price from the product catalog.

**Requirements:**
- Calculate sales at item level before combining products' item values.
- Use all item occurrences and leave the source quantities and prices unchanged.

**Available datasets and columns:**
**`order_items:`** item_id, order_id, product_id, quantity, unit_price

**Expected output:** A Pandas DataFrame with `product_id` and `sales`, one row per represented non-missing product ID. `sales` is the sum of quantity-times-unit-price across that product's items.

**Solution:**


In [ ]:
result = order_items.assign(line_total=order_items['quantity']*order_items['unit_price']).groupby('product_id',as_index=False).agg(sales=('line_total','sum'))

display(result)


### Q29 — Conditional aggregation

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Count `completed` and `cancelled` orders for each non-missing `customer_id` in `orders`. Return `customer_id`, `completed`, and `cancelled` as counts. Include customers represented only by other statuses with zero in both count columns.

**Requirements:**
- Match the two status labels exactly without filtering away customers represented by other statuses.
- Each order contributes at most one to either count; amounts do not affect the calculation.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with `customer_id`, `completed`, and `cancelled`, one row per observed non-missing customer ID. Both measure columns are counts, with zero where that customer has no orders of the corresponding status.

**Solution:**


In [ ]:
result = orders.assign(completed=orders['status'].eq('completed').astype(int),cancelled=orders['status'].eq('cancelled').astype(int)).groupby('customer_id',as_index=False)[['completed','cancelled']].sum()

display(result)


### Q30 — Pivot revenue

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Create a wide revenue summary from `orders` with one row per `customer_id` and one column per observed `status`. Each customer-status cell should contain the sum of `amount`, with zero where that combination is absent. Keep `customer_id` as a regular output column.

**Requirements:**
- Sum amounts when a customer-status combination has multiple orders.
- Use non-missing customer and status keys, and fill absent combinations with zero rather than omitting their cells.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A wide Pandas DataFrame with `customer_id` as a regular column and a revenue column for each observed non-missing status. Each customer appears once; status cells contain summed amounts or zero for absent combinations.

**Solution:**


In [ ]:
result = orders.pivot_table(index='customer_id',columns='status',values='amount',aggfunc='sum',fill_value=0).reset_index()

display(result)


## 4. Merges, Joins & Reshaping


### Q31 — Customer order join

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Enrich `orders` with `name` and `segment` from `customers`, matching on `customer_id`. Retain all original order columns and every order, leaving the added fields missing when no customer matches. Require the customer lookup to contain at most one matching record per key.

**Requirements:**
- Enforce the many-orders-to-one-customer lookup contract; duplicate customer lookup keys must not silently multiply orders.
- Add only `name` and `segment`, without modifying either source.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount
**`customers:`** customer_id, name, email, segment, country, signup_date

**Expected output:** A Pandas DataFrame with all order columns plus `name` and `segment`, one row per input order. Unmatched orders remain with missing values in the added columns.

**Solution:**


In [ ]:
result = orders.merge(customers[['customer_id','name','segment']],on='customer_id',how='left',validate='many_to_one')

display(result)


### Q32 — Customer order counts

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Attach `order_count` to every row of `customers`, counting matching records in `orders` by `customer_id`. Include orders of every status and use zero for customers without orders. Retain all original customer columns, row order, and index values.

**Requirements:**
- Count all matching order records, including repeated orders and non-completed statuses.
- Keep customers with no match and store their count as integer zero.

**Available datasets and columns:**
**`customers:`** customer_id, name, email, segment, country, signup_date
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame containing all original customer columns plus integer `order_count`. Every customer row remains in its original position with its original index.

**Solution:**


In [ ]:
counts = orders.groupby('customer_id').size().rename('order_count'); result = customers.join(counts,on='customer_id').assign(order_count=lambda x:x['order_count'].fillna(0).astype(int))

display(result)


### Q33 — Order line enrichment

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Add `order_date` and `status` from `orders` to each row of `order_items`, matching on `order_id`. Preserve every item and all its original columns, leaving the added fields missing for unmatched items. Require at most one order record per matching key.

**Requirements:**
- Check the many-items-to-one-order lookup relationship rather than allowing duplicate order keys to multiply rows.
- Do not filter items by order status or add unrelated order fields.

**Available datasets and columns:**
**`order_items:`** item_id, order_id, product_id, quantity, unit_price
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with all order-item columns plus `order_date` and `status`. There is one row per item; unmatched items have missing values in both added fields.

**Solution:**


In [ ]:
result = order_items.merge(orders[['order_id','order_date','status']],on='order_id',how='left',validate='many_to_one')

display(result)


### Q34 — Product line enrichment

**Interview scenario:** You are building an e-commerce order-line fact pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Enrich `order_items` with `category` from `products`, matching on `product_id`. Keep all item rows and original item columns, with a missing category where no product matches. Require at most one product record per matching key.

**Requirements:**
- Enforce at most one product lookup row per key.
- Enrich items without replacing their `unit_price` or adding the catalog price.

**Available datasets and columns:**
**`order_items:`** item_id, order_id, product_id, quantity, unit_price
**`products:`** product_id, sku, category, price

**Expected output:** A Pandas DataFrame with all original order-item columns plus `category`, one row per input item. Items without a product match remain with missing category values.

**Solution:**


In [ ]:
result = order_items.merge(products[['product_id','category']],on='product_id',how='left',validate='many_to_one')

display(result)


### Q35 — Revenue by category

**Interview scenario:** You are building an e-commerce order-line fact pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Calculate revenue by product category from `order_items` and `products`. Match items to categories using `product_id`, then total each item's quantity multiplied by its sale-time unit price. Return `category` and `revenue`, excluding items with no non-missing category from the grouped result.

**Requirements:**
- Use the item's `unit_price` rather than `products.price`.
- Combine all products within each non-missing category and omit items that cannot be assigned such a category.

**Available datasets and columns:**
**`order_items:`** item_id, order_id, product_id, quantity, unit_price
**`products:`** product_id, sku, category, price

**Expected output:** A Pandas DataFrame with `category` and `revenue`, one row per represented non-missing category. Revenue is the sum of its items' quantity-times-sale-price values.

**Solution:**


In [ ]:
x=order_items.merge(products[['product_id','category']],on='product_id',how='left'); result=x.assign(revenue=x['quantity']*x['unit_price']).groupby('category',as_index=False)['revenue'].sum()

display(result)


### Q36 — Unmatched customer keys

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Identify rows in `orders` whose `customer_id` has no match in `customers`. Return the qualifying order rows with their original columns, excluding any temporary match-indicator fields. Compare customer keys rather than customer names.

**Requirements:**
- Check membership by customer ID without inferring matches from names or other attributes.
- Return complete unmatched orders, not a distinct-key list; remove temporary match-indicator columns.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount
**`customers:`** customer_id, name, email, segment, country, signup_date

**Expected output:** A Pandas DataFrame containing all original order columns for orders with unmatched customer IDs. No customer attributes or match indicators are included; a fully matched input gives an empty result.

**Solution:**


In [ ]:
x=orders.merge(customers[['customer_id']],on='customer_id',how='left',indicator=True);result=x.loc[x['_merge'].eq('left_only')].drop(columns='_merge')

display(result)


### Q37 — Anti-join customers

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Return customer records from `customers` whose `customer_id` does not appear in `orders`. Include all original customer columns and retain the qualifying records' input order and index values. A customer with any order status is considered to have ordered.

**Requirements:**
- Any matching order establishes membership, regardless of its status or amount.
- Leave the source customer records unchanged and retain duplicate qualifying records.

**Available datasets and columns:**
**`customers:`** customer_id, name, email, segment, country, signup_date
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame of complete customers with no matching order ID reference, preserving their original row order and index. It contains no order columns or count fields.

**Solution:**


In [ ]:
result=customers.loc[~customers['customer_id'].isin(orders['customer_id'])]

display(result)


### Q38 — Wide-to-long

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Melt `orders` from wide to long format for the `status` and `amount` columns. Return `order_id`, `attribute`, and `value`, with two rows per input order: one describing its status and one its amount. Do not carry the other order columns into the result.

**Requirements:**
- Use the wide-to-long transformation required by the Task.
- Retain both attribute entries even when their source value is missing; do not deduplicate repeated order records.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with exactly `order_id`, `attribute`, and `value`. Each input order contributes two rows, labelled `status` and `amount`, containing the corresponding original values.

**Solution:**


In [ ]:
result=orders[['order_id','status','amount']].melt(id_vars='order_id',var_name='attribute',value_name='value')

display(result)


### Q39 — Pivot count by segment

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Combine `orders` with customer `segment` from `customers` using `customer_id`. Build a wide table with one row per segment and one column per observed order status. Populate cells with counts of non-missing `order_id`, fill absent combinations with zero, and keep `segment` as a regular column.

**Requirements:**
- Count non-missing order IDs rather than summing amounts or counting distinct customers.
- Exclude missing segment or status grouping keys; use zero for absent combinations among the retained groups.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount
**`customers:`** customer_id, name, email, segment, country, signup_date

**Expected output:** A wide Pandas DataFrame with `segment` as a regular column and one count column per observed non-missing status. Each non-missing segment appears once, with counts of non-missing order IDs and zero-filled absent combinations.

**Solution:**


In [ ]:
x=orders.merge(customers[['customer_id','segment']],on='customer_id',how='left');result=x.pivot_table(index='segment',columns='status',values='order_id',aggfunc='count',fill_value=0).reset_index()

display(result)


### Q40 — Join cardinality check

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Check row-count preservation when enriching `orders` with customer `name` on `customer_id`, retaining unmatched orders. Require at most one customer match per key. Return a one-row DataFrame with `before`, `after`, and `difference`, where difference is the joined row count minus the original order count.

**Requirements:**
- Validate customer-key uniqueness as part of the join; a multiplied result is not an acceptable successful check.
- Retain unmatched orders when measuring the joined row count.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount
**`customers:`** customer_id, name, email, segment, country, signup_date

**Expected output:** A one-row Pandas DataFrame with integer columns `before`, `after`, and `difference`. For a valid many-to-one enrichment, both counts equal the input order count and `difference` is zero.

**Solution:**


In [ ]:
joined=orders.merge(customers[['customer_id','name']],on='customer_id',how='left',validate='many_to_one');result=pd.DataFrame([{'before':len(orders),'after':len(joined),'difference':len(joined)-len(orders)}])

display(result)


## 5. Dates, Time Series & Windows


### Q41 — Parse order dates

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Convert `order_date` in a new version of `orders` to a Pandas datetime dtype. Retain every row, index value, and other column, and leave the original DataFrame unchanged. Display the resulting date column's dtype along with the transformed data.

**Requirements:**
- The converted `order_date` column must have a Pandas datetime dtype, not an object or string dtype containing date-like values.
- Preserve row order, index values, column order, and all non-date values; leave the input `orders` DataFrame unchanged.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A new `orders` DataFrame with the original rows, columns, and index, with `order_date` converted to datetime. Display both the transformed DataFrame and the resulting `order_date` dtype. The dtype report must describe the converted column, not the input column.

**Solution:**


In [ ]:
result=orders.assign(order_date=pd.to_datetime(orders['order_date']))

display(result)
print(result['order_date'].dtype)


### Q42 — Monthly revenue

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Calculate revenue by calendar month from `orders`, including every status. Derive `month` from `order_date` and return one row per represented year-month with `month` and the total `amount`. Do not combine the same month across different years.

**Requirements:**
- Keep each calendar year-month distinct and include amounts from every status.
- Represent the grouping key as a monthly Pandas period without changing `orders`.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with `month` and `amount`, one row per represented calendar month. `month` contains monthly period values and `amount` contains total order revenue.

**Solution:**


In [ ]:
x=orders.assign(month=pd.to_datetime(orders['order_date']).dt.to_period('M'));result=x.groupby('month',as_index=False)['amount'].sum()

display(result)


### Q43 — Daily transaction volume

**Interview scenario:** You are building a multi-currency payment ingestion service. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Count transactions by the UTC calendar date of `transaction_ts` in `transactions`. Return one row per represented date with `day` and the transaction count as `transactions`. Count records, regardless of transaction type or amount.

**Requirements:**
- Derive dates in UTC before assigning records to days.
- Count transaction records, including repeated IDs, without filtering by type or amount.

**Available datasets and columns:**
**`transactions:`** transaction_id, customer_id, amount, currency, transaction_ts, type

**Expected output:** A Pandas DataFrame with `day` and integer `transactions`, one row per represented UTC date. `day` contains calendar-date values; the second column is that day's record count.

**Solution:**


In [ ]:
x=transactions.assign(day=pd.to_datetime(transactions['transaction_ts'],utc=True).dt.date);result=x.groupby('day',as_index=False).size().rename(columns={'size':'transactions'})

display(result)


### Q44 — Latest order per customer

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** From `orders`, return one complete order row for each `customer_id`. Select the row with the most recent `order_date`; if dates match for the same customer, retain the row with the largest `order_id`. Interpret `order_date` as a datetime and retain all original order columns.

**Requirements:**
- Select the entire winning order rather than mixing values from different orders.
- Break a customer's date ties by the largest order ID and leave the source unchanged.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame containing one complete latest order per customer ID, with all original columns and datetime `order_date` values. The selected rows retain their original indexes.

**Solution:**


In [ ]:
x=orders.assign(order_date=pd.to_datetime(orders['order_date']));result=x.sort_values(['order_date','order_id']).drop_duplicates('customer_id',keep='last')

display(result)


### Q45 — Running customer spend

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Add `running_spend` to every row of `orders`, accumulating `amount` separately for each customer. Process orders by `order_date` ascending and then `order_id` ascending, including the current row in its total. Return all original order columns with parsed datetime dates, sorted by customer and that sequence.

**Requirements:**
- Accumulate within each customer only, including orders of every status.
- Use the order-ID tie-breaker for same-date orders so each running total corresponds to the stated sequence.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with all order columns and `running_spend`, one row per order. Dates are datetime values; rows are ordered by customer ID, order date, and order ID ascending, with totals including the current order.

**Solution:**


In [ ]:
x=orders.assign(order_date=pd.to_datetime(orders['order_date'])).sort_values(['customer_id','order_date','order_id']);result=x.assign(running_spend=x.groupby('customer_id')['amount'].cumsum())

display(result)


### Q46 — Previous order amount

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Add `previous_amount` to each row in `orders`, containing the amount of the preceding order from the same customer. Order each customer's history by `order_date` ascending and then `order_id` ascending. Retain every order and original column, with missing previous amounts for first orders, sorted by customer and this sequence.

**Requirements:**
- Use the immediately preceding row in the customer's ordered history, not the preceding row from another customer.
- Keep first orders in the result with missing `previous_amount`.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with all order columns plus `previous_amount`, with `order_date` parsed as datetime. Rows are sorted by customer ID, date, and order ID ascending, and the first order per customer has no previous amount.

**Solution:**


In [ ]:
x=orders.assign(order_date=pd.to_datetime(orders['order_date'])).sort_values(['customer_id','order_date','order_id']);result=x.assign(previous_amount=x.groupby('customer_id')['amount'].shift())

display(result)


### Q47 — Days between orders

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Calculate the number of calendar days since each customer's preceding order in `orders`. Sequence records by `order_date` ascending and `order_id` ascending within each customer. Return all order columns plus `days_since_previous`, leaving it missing for first orders and sorting by customer and this sequence.

**Requirements:**
- Calculate gaps within a customer's history, not across customer boundaries.
- Orders on the same date have a zero-day gap; first orders remain with missing gaps.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame containing all order columns and `days_since_previous`, with datetime order dates. Rows follow customer ID, order date, and order ID ascending; gaps are measured in days.

**Solution:**


In [ ]:
x=orders.assign(order_date=pd.to_datetime(orders['order_date'])).sort_values(['customer_id','order_date','order_id']);result=x.assign(days_since_previous=x.groupby('customer_id')['order_date'].diff().dt.days)

display(result)


### Q48 — Monthly MoM growth

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Summarize total `orders.amount` by the year-month of `order_date`, including all statuses. Return chronological rows with `month`, `amount`, and `mom_pct`, comparing each represented month with the preceding represented month. Express change as a percentage of the prior amount; the first month has no prior comparison.

**Requirements:**
- Compare adjacent represented months rather than adding absent calendar months.
- Express relative change on a percentage scale and do not round intermediate totals.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with `month`, `amount`, and `mom_pct`, sorted chronologically. `month` is a monthly period, `amount` is total revenue, and the first row's percentage change is missing.

**Solution:**


In [ ]:
x=orders.assign(month=pd.to_datetime(orders['order_date']).dt.to_period('M')).groupby('month',as_index=False)['amount'].sum().sort_values('month');result=x.assign(mom_pct=x['amount'].pct_change()*100)

display(result)


### Q49 — Seven-day rolling average

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Calculate daily revenue from `orders.amount` using `order_date`. Include every calendar date between the earliest and latest order dates, assigning zero revenue to dates without orders. Return `day` and `rolling_avg_7d`, averaging the current day and preceding six days. For the initial dates, average only the days available since the series began.

**Requirements:**
- Calculate one revenue series across all customers and statuses, not separate customer histories.
- Include zero-revenue dates in the average; initial windows use only dates available since the series starts.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with datetime `day` and numeric `rolling_avg_7d`, in chronological order. Every date in the input's date range appears once, including dates with no orders.

**Solution:**


In [ ]:
x=orders.assign(day=pd.to_datetime(orders['order_date'])).groupby('day')['amount'].sum().sort_index().asfreq('D',fill_value=0);result=x.rolling(7,min_periods=1).mean().rename('rolling_avg_7d').reset_index()

display(result)


### Q50 — Timezone conversion

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Add `event_ts_local` to `events` by interpreting `event_ts` as UTC and converting it to the `America/Vancouver` timezone. The new values should remain timezone-aware and reflect daylight-saving offsets where applicable. Preserve all existing rows, columns, and index values, including the original event timestamps.

**Requirements:**
- Convert timezone without changing the instant represented by each timestamp.
- Retain timezone information and leave the original `event_ts` values untouched.

**Available datasets and columns:**
**`events:`** event_id, user_id, event_type, event_ts, session_id, properties

**Expected output:** A Pandas DataFrame containing every original event column plus timezone-aware `event_ts_local` values in `America/Vancouver`. Row count, row order, and index remain unchanged.

**Solution:**


In [ ]:
result=events.assign(event_ts_local=pd.to_datetime(events['event_ts'],utc=True).dt.tz_convert('America/Vancouver'))

display(result)


## 6. Cleaning, Validation & Deduplication


### Q51 — Normalize emails

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Create `email_normalized` in a copy of `customers` by trimming surrounding whitespace from `email` and converting it to lowercase. Preserve missing emails as missing values. Retain all original rows, columns, and indexes, including the unchanged source email column.

**Requirements:**
- Normalize into the added column, not over the original email field.
- Whitespace-only emails become empty strings; missing emails remain missing rather than becoming the text `nan` or `None`.

**Available datasets and columns:**
**`customers:`** customer_id, name, email, segment, country, signup_date

**Expected output:** A Pandas DataFrame with every original customer column plus `email_normalized`, containing lowercase trimmed email text or missing values. All rows, their order, and their indexes are retained.

**Solution:**


In [ ]:
result=customers.assign(email_normalized=customers['email'].astype('string').str.strip().str.lower())

display(result)


### Q52 — Find duplicate emails

**Interview scenario:** You are building a customer-data cleansing pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Find all records in `customers_dirty` whose normalized email occurs more than once. Normalize by trimming surrounding whitespace and lowercasing, and treat repeated missing values as a duplicate group. Return every record in each duplicate group with its original columns and `email_normalized`, sorted by normalized email.

**Requirements:**
- Retain all members of repeated-email groups, not just occurrences after the first.
- Include repeated missing normalized values and leave original email text unchanged.

**Available datasets and columns:**
**`customers_dirty:`** customer_id, name, email, segment, country, signup_date

**Expected output:** A Pandas DataFrame with all dirty-customer columns plus `email_normalized`, containing only groups with more than one record. Sort by normalized email ascending with missing values last; no extra within-group tie-breaker is required.

**Solution:**


In [ ]:
x=customers_dirty.assign(email_normalized=customers_dirty['email'].astype('string').str.strip().str.lower());result=x.loc[x.duplicated('email_normalized',keep=False)].sort_values('email_normalized')

display(result)


### Q53 — Keep latest CDC record

**Interview scenario:** You are building a change-data-capture replication pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Return the latest change record for each `customer_id` in `cdc_changes`. Interpret `changed_at` as a UTC timestamp, choosing the largest `change_id` when timestamps tie. Retain all original columns of selected records, including the parsed timestamp, without excluding delete operations.

**Requirements:**
- Resolve equal timestamps by largest `change_id`, not operation type.
- Select the latest change before interpreting deletion; delete records are valid output here.

**Available datasets and columns:**
**`cdc_changes:`** change_id, customer_id, operation, changed_at, email

**Expected output:** A Pandas DataFrame with one complete CDC record per customer ID and all original CDC columns. `changed_at` contains UTC-aware datetime values, and the latest operation may be a delete.

**Solution:**


In [ ]:
x=cdc_changes.assign(changed_at=pd.to_datetime(cdc_changes['changed_at'],utc=True));result=x.sort_values(['changed_at','change_id']).drop_duplicates('customer_id',keep='last')

display(result)


### Q54 — Missing customer emails

**Interview scenario:** You are building a customer-data cleansing pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Return complete rows from `customers_dirty` whose `email` is missing or becomes empty after surrounding whitespace is removed. Preserve the original email values in the returned rows rather than replacing them with trimmed values. Retain the qualifying rows' order and index.

**Requirements:**
- Use trimming only to identify blank values; do not overwrite the source email column.
- This check finds absent or blank emails, not every malformed nonblank address.

**Available datasets and columns:**
**`customers_dirty:`** customer_id, name, email, segment, country, signup_date

**Expected output:** A Pandas DataFrame of complete dirty-customer rows with missing or whitespace-only emails, retaining the original values, row order, and index. No normalized-email helper column is included.

**Solution:**


In [ ]:
x=customers_dirty['email'].astype('string').str.strip();result=customers_dirty.loc[x.isna()|x.eq('')]

display(result)


### Q55 — Validate positive amounts

**Interview scenario:** You are building a multi-currency payment ingestion service. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Find transactions in `transactions` whose `amount` is missing, zero, or negative. Return the complete qualifying rows with their original order and indexes. A strictly positive amount is valid for this check, regardless of transaction type.

**Requirements:**
- Zero fails the strictly-positive check, as do missing and negative amounts.
- Do not excuse negative amounts based on transaction type or modify the source records.

**Available datasets and columns:**
**`transactions:`** transaction_id, customer_id, amount, currency, transaction_ts, type

**Expected output:** A Pandas DataFrame of complete invalid-amount transactions, with all original columns and the selected rows' original order and indexes. Strictly positive amounts are excluded.

**Solution:**


In [ ]:
result=transactions.loc[transactions['amount'].isna()|transactions['amount'].le(0)]

display(result)


### Q56 — Duplicate events

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Find all records in `events_duplicates` belonging to a repeated combination of `user_id`, `event_type`, and `event_ts`. Return every occurrence in each duplicate group, including the first occurrence, with all original fields. Preserve the matching rows' input order and indexes.

**Requirements:**
- Use the three-field logical key, not `event_id`, to identify repeated events.
- Do not select a single survivor: every occurrence in each repeated-key group must be returned.

**Available datasets and columns:**
**`events_duplicates:`** event_id, user_id, event_type, event_ts, session_id, properties

**Expected output:** A Pandas DataFrame with all original event columns for repeated logical keys, including each group's first occurrence. Selected rows keep their input order and original indexes.

**Solution:**


In [ ]:
result=events_duplicates.loc[events_duplicates.duplicated(['user_id','event_type','event_ts'],keep=False)]

display(result)


### Q57 — Deduplicate event feed

**Interview scenario:** You are building a production data engineering pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Deduplicate `events_duplicates` by the logical event key `user_id`, `event_type`, and `event_ts`. For each key, retain the record with the highest `event_id`. Return one complete event record per key, keeping the selected records' original index values.

**Requirements:**
- Deduplicate on `user_id`, `event_type`, and `event_ts` together.
- Keep the highest-ID record intact rather than aggregating fields across duplicate events.

**Available datasets and columns:**
**`events_duplicates:`** event_id, user_id, event_type, event_ts, session_id, properties

**Expected output:** A Pandas DataFrame containing one complete event row per logical key, chosen by maximum `event_id`. Original columns and the surviving records' index values are retained.

**Solution:**


In [ ]:
result=events_duplicates.sort_values('event_id').drop_duplicates(['user_id','event_type','event_ts'],keep='last')

display(result)


### Q58 — Invalid dates

**Interview scenario:** You are building a customer-data cleansing pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Identify records in `customers_dirty` whose `signup_date` cannot be converted to a datetime, including missing dates. Return the complete original records without replacing the source date values. Retain the qualifying records' order and indexes.

**Requirements:**
- Retain missing dates as failures alongside unparseable values.
- Inspect parseability without replacing original date text or adding a parsed-date column to the result.

**Available datasets and columns:**
**`customers_dirty:`** customer_id, name, email, segment, country, signup_date

**Expected output:** A Pandas DataFrame of complete dirty-customer records whose signup date is missing or cannot be parsed. The result preserves their input order, indexes, and original `signup_date` values.

**Solution:**


In [ ]:
result=customers_dirty.loc[pd.to_datetime(customers_dirty['signup_date'],errors='coerce').isna()]

display(result)


### Q59 — Reconciliation check

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Reconcile each `orders.amount` with its total item value from `order_items`, calculated from quantity and sale-time unit price. Return original order columns plus `line_total` only where the absolute difference exceeds 0.01. Use zero for a missing item total during comparison, but leave the returned `line_total` missing for orders without items.

**Requirements:**
- Use sale-time item prices and the strict 0.01 tolerance without rounding before comparison.
- Substitute zero only for the comparison; do not fill the returned missing item totals.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount
**`order_items:`** item_id, order_id, product_id, quantity, unit_price

**Expected output:** A Pandas DataFrame with all order columns plus `line_total` for mismatches only. Orders without items retain missing `line_total` values if they qualify; differences of exactly 0.01 are excluded.

**Solution:**


In [ ]:
lines=order_items.assign(line_total=order_items['quantity']*order_items['unit_price']).groupby('order_id',as_index=False)['line_total'].sum();x=orders.merge(lines,on='order_id',how='left');result=x.loc[(x['amount']-x['line_total'].fillna(0)).abs().gt(0.01)].copy()

display(result)


### Q60 — Schema audit

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Audit the schema and missing values of `orders`. Return a DataFrame with one row per input column and fields `column`, `dtype`, and `missing`. Represent the dtype as text, count missing values in that column, and retain the original column order in the report.

**Requirements:**
- Report the DataFrame's existing dtypes rather than converting columns.
- Count missing cells independently for each column, including columns with zero missing values.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with `column`, `dtype`, and integer `missing`, one row per `orders` column in original column order. Dtypes are text and the source data is unchanged.

**Solution:**


In [ ]:
result=pd.DataFrame({'column':orders.columns,'dtype':orders.dtypes.astype(str).values,'missing':orders.isna().sum().values})

display(result)


## 7. Nested JSON, Events & Logs


### Q61 — Flatten event properties

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Expose the nested event properties in `events` as additional columns, including `property_page` and `property_device` from `properties.page` and `properties.device`. Prefix extracted property names with `property_`. Retain every event, its index, and all original columns, including the nested `properties` column.

**Requirements:**
- Extract properties into additional columns without removing the original nested object.
- Keep each property's values aligned with its event and retain missing values where a property is absent.

**Available datasets and columns:**
**`events:`** event_id, user_id, event_type, event_ts, session_id, properties

**Expected output:** A Pandas DataFrame with all original event columns, including `properties`, plus `property_`-prefixed property columns such as `property_page` and `property_device`. Every event and its original index remain present.

**Solution:**


In [ ]:
result=events.join(pd.json_normalize(events['properties']).add_prefix('property_'))

display(result)


### Q62 — Event type counts

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Count records in `events` for each non-missing `event_type`. Return a DataFrame with one row per type and columns `event_type` and `events`. Count event rows rather than distinct users or sessions.

**Requirements:**
- Count repeated event records separately rather than deduplicating IDs.
- Exclude missing event types and leave the event data unchanged.

**Available datasets and columns:**
**`events:`** event_id, user_id, event_type, event_ts, session_id, properties

**Expected output:** A Pandas DataFrame with `event_type` and integer `events`, one row per observed non-missing event type. Each count is the number of event records of that type.

**Solution:**


In [ ]:
result=events.groupby('event_type',as_index=False).size().rename(columns={'size':'events'})

display(result)


### Q63 — Distinct active users

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Calculate daily active users from `events`, deriving each UTC calendar `day` from `event_ts`. Return `day` and the count of distinct non-missing `user_id` values as `dau`. Multiple events by the same user on a date should contribute only one to that day's count.

**Requirements:**
- Count a user once per UTC day, even across multiple event types or sessions.
- Exclude missing user IDs from distinct-user counts, not valid repeated users from other days.

**Available datasets and columns:**
**`events:`** event_id, user_id, event_type, event_ts, session_id, properties

**Expected output:** A Pandas DataFrame with calendar-date `day` and integer `dau`, one row per represented UTC day. A user can contribute once to each day's count.

**Solution:**


In [ ]:
x=events.assign(day=pd.to_datetime(events['event_ts'],utc=True).dt.date);result=x.groupby('day',as_index=False)['user_id'].nunique().rename(columns={'user_id':'dau'})

display(result)


### Q64 — First event per user

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Select the earliest event for each `user_id` in `events`, comparing UTC timestamps from `event_ts`. Break equal-timestamp ties with the smallest `event_id`. Return each selected event's original columns and its parsed timestamp as `ts`.

**Requirements:**
- Select the entire earliest record; resolve timestamp ties by smallest event ID.
- Keep the original timestamp column and add the UTC-aware parsed value separately.

**Available datasets and columns:**
**`events:`** event_id, user_id, event_type, event_ts, session_id, properties

**Expected output:** A Pandas DataFrame with one selected event per user, all original event columns, and UTC-aware `ts`. Each row is that user's earliest event under the timestamp and event-ID ordering.

**Solution:**


In [ ]:
x=events.assign(ts=pd.to_datetime(events['event_ts'],utc=True));result=x.sort_values(['ts','event_id']).drop_duplicates('user_id',keep='first')

display(result)


### Q65 — Session lengths

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Calculate session lengths from `events`, grouping by `session_id`. Return `session_id`, the earliest timestamp as `first`, the latest as `last`, event count as `events`, and elapsed `duration_seconds`. Interpret timestamps in UTC and calculate duration from the first event to the last; a single-event session has zero duration.

**Requirements:**
- Measure elapsed time within each session, not across all events or separate user histories.
- Count event records, including repeated event IDs, rather than distinct users.

**Available datasets and columns:**
**`events:`** event_id, user_id, event_type, event_ts, session_id, properties

**Expected output:** A Pandas DataFrame with `session_id`, UTC-aware `first` and `last`, integer `events`, and numeric `duration_seconds`. Each row summarizes one session; single-event sessions have zero duration.

**Solution:**


In [ ]:
x=events.assign(ts=pd.to_datetime(events['event_ts'],utc=True));result=x.groupby('session_id',as_index=False).agg(first=('ts','min'),last=('ts','max'),events=('event_id','size'));result['duration_seconds']=(result['last']-result['first']).dt.total_seconds()

display(result)


### Q66 — Pipeline error logs

**Interview scenario:** You are building an ETL observability system. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Return complete records from `pipeline_logs` whose `level` equals `ERROR`, ignoring letter case. Preserve all original fields, row order, and index values. This check uses the level field rather than searching the message text.

**Requirements:**
- Use a case-insensitive exact level comparison, not a substring search.
- Preserve the original level text and leave all source records unchanged.

**Available datasets and columns:**
**`pipeline_logs:`** log_id, pipeline, level, duration_ms, timestamp, message

**Expected output:** A Pandas DataFrame of complete error-level log records, retaining their original columns, values, row order, and index. Message contents do not determine membership.

**Solution:**


In [ ]:
result=pipeline_logs.loc[pipeline_logs['level'].astype('string').str.upper().eq('ERROR')]

display(result)


### Q67 — Pipeline runtime percentiles

**Interview scenario:** You are building an ETL observability system. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Summarize `duration_ms` separately for each `pipeline` in `pipeline_logs`. Return `pipeline`, the median as `p50_ms`, and the 95th percentile as `p95_ms`. Use linear interpolation for percentiles between observed values and exclude missing durations from the calculations.

**Requirements:**
- Calculate each pipeline's percentiles from its own non-missing durations.
- Use linear interpolation, without selecting an arbitrary nearest observed runtime.

**Available datasets and columns:**
**`pipeline_logs:`** log_id, pipeline, level, duration_ms, timestamp, message

**Expected output:** A Pandas DataFrame with `pipeline`, `p50_ms`, and `p95_ms`, one row per represented pipeline. Both measures are durations in milliseconds and may be interpolated values.

**Solution:**


In [ ]:
result=pipeline_logs.groupby('pipeline',as_index=False).agg(p50_ms=('duration_ms','median'),p95_ms=('duration_ms',lambda s:s.quantile(.95)))

display(result)


### Q68 — Error rate by pipeline

**Interview scenario:** You are building an ETL observability system. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Measure error frequency for each `pipeline` in `pipeline_logs`. Return `pipeline`, `log_count`, and `error_rate_pct`. Count all log records and calculate the percentage whose `level` equals `ERROR`, ignoring letter case. Include pipelines with no errors at zero percent.

**Requirements:**
- Evaluate the level field case-insensitively rather than searching messages.
- Use each pipeline's log population for its rate and include pipelines that have no errors.

**Available datasets and columns:**
**`pipeline_logs:`** log_id, pipeline, level, duration_ms, timestamp, message

**Expected output:** A Pandas DataFrame with `pipeline`, integer `log_count`, and numeric `error_rate_pct`, one row per pipeline. The rate is the percentage of its log records classified as errors, with zero for pipelines without errors.

**Solution:**


In [ ]:
x=pipeline_logs.assign(is_error=pipeline_logs['level'].astype('string').str.upper().eq('ERROR').fillna(False));result=x.groupby('pipeline',as_index=False).agg(log_count=('log_id','size'),error_rate_pct=('is_error',lambda s:s.mean()*100)).astype({'error_rate_pct':'Float64'})

display(result)


### Q69 — Slow pipeline records

**Interview scenario:** You are building an ETL observability system. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Find unusually slow records in `pipeline_logs` using the 95th percentile of `duration_ms` across all pipelines as the threshold. Use linear interpolation for the percentile and return complete rows with durations strictly above it. Preserve their input order and indexes.

**Requirements:**
- Use one threshold across all pipelines, not a separate percentile per pipeline.
- Exclude missing durations and records equal to the threshold; retain original log fields.

**Available datasets and columns:**
**`pipeline_logs:`** log_id, pipeline, level, duration_ms, timestamp, message

**Expected output:** A Pandas DataFrame of complete log records with durations strictly above the overall 95th-percentile threshold. Qualifying rows retain their input order and index; the threshold is not an added output column.

**Solution:**


In [ ]:
result=pipeline_logs.loc[pipeline_logs['duration_ms'].gt(pipeline_logs['duration_ms'].quantile(.95))]

display(result)


### Q70 — Event funnel summary

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Produce a three-row summary of distinct users in `events` for `event_type` values `view`, `cart`, and `purchase`, in that order. Return `event_type` and `unique_users`. Count a user separately in each type they appear in and use zero for an absent event type.

**Requirements:**
- Count distinct non-missing users independently for each event type.
- Do not require users to have passed through earlier funnel stages or exclude users appearing in multiple stages.

**Available datasets and columns:**
**`events:`** event_id, user_id, event_type, event_ts, session_id, properties

**Expected output:** A three-row Pandas DataFrame with `event_type` and integer `unique_users`, ordered `view`, `cart`, `purchase`. Missing event types have zero counts; other event types are omitted.

**Solution:**


In [ ]:
result=events.groupby('event_type')['user_id'].nunique().reindex(['view','cart','purchase'],fill_value=0).rename('unique_users').reset_index()

display(result)


## 8. Production Data Engineering Scenarios


### Q71 — Incremental watermark

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Select rows from `orders` whose `order_date` is strictly later than September 1, 2025. Interpret the field as a date or datetime for the comparison and exclude orders on the watermark date. Return complete original rows with their original order and index values.

**Requirements:**
- Apply the date comparison without overwriting the original `order_date` values.
- Use the strict watermark boundary and retain all fields of qualifying orders.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame of complete orders later than September 1, 2025, in their original order and with original indexes. Dates remain in their source representation, and orders on the boundary are excluded.

**Solution:**


In [ ]:
result=orders.loc[pd.to_datetime(orders['order_date']).gt(pd.Timestamp('2025-09-01'))]

display(result)


### Q72 — Idempotent batch merge

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Treat the first 100 rows and the first 150 rows of `orders` as overlapping input batches. Combine them into one result with a single record per `order_id`, retaining the last encountered record when IDs repeat. Keep every original order column and do not modify the source DataFrame.

**Requirements:**
- Process the 100-row batch before the 150-row batch when determining the last occurrence.
- Deduplicate by `order_id`, not by equality of all fields, and preserve the source DataFrame.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame of complete order records with one row per order ID across the two batches. Later occurrences replace earlier ones; on the supplied data, the records match the first 150 orders.

**Solution:**


In [ ]:
a=orders.head(100);b=orders.head(150);result=pd.concat([a,b],ignore_index=True).drop_duplicates('order_id',keep='last')

display(result)


### Q73 — CDC latest state

**Interview scenario:** You are building a change-data-capture replication pipeline. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Build current customer state from `cdc_changes`. Select the latest `changed_at` per `customer_id`, breaking ties with the largest `change_id`, then exclude customers whose selected operation is `DELETE` or `D`, ignoring case. Return the selected original change fields and parsed UTC timestamp `ts`; do not resurrect an earlier non-delete change.

**Requirements:**
- Determine the latest change before filtering delete operations.
- Match `DELETE` and `D` case-insensitively and retain original change fields without reconstructing earlier customer records.

**Available datasets and columns:**
**`cdc_changes:`** change_id, customer_id, operation, changed_at, email

**Expected output:** A Pandas DataFrame with at most one row per customer, containing all original CDC columns plus UTC-aware `ts`. Customers whose latest operation is a delete are absent.

**Solution:**


In [ ]:
x=cdc_changes.assign(ts=pd.to_datetime(cdc_changes['changed_at'],utc=True)).sort_values(['ts','change_id']).drop_duplicates('customer_id',keep='last');result=x.loc[~x['operation'].astype('string').str.upper().isin(['DELETE','D'])]

display(result)


### Q74 — Revenue anomaly flags

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Add a Boolean `is_anomaly` column to `orders`, flagging amounts strictly greater than the mean amount plus twice its sample standard deviation. Calculate both statistics across the order amounts, excluding missing values. Keep every original row, column, and index; amounts equal to the threshold are not anomalies.

**Requirements:**
- Use the sample standard deviation across all order amounts, not separate customer or status statistics.
- Add a flag to every row rather than filtering to anomalies; leave `orders` unchanged.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with all original order columns plus Boolean `is_anomaly`, preserving row order and index. Only amounts strictly above the stated global threshold are flagged.

**Solution:**


In [ ]:
threshold=orders['amount'].mean()+2*orders['amount'].std();result=orders.assign(is_anomaly=orders['amount'].gt(threshold))

display(result)


### Q75 — Currency normalization

**Interview scenario:** You are building a multi-currency payment ingestion service. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Add `amount_cad` to `transactions` using illustrative rates of 1.35 for USD, 1.00 for CAD, 1.48 for EUR, and 1.72 for GBP. Multiply `amount` by the rate for its `currency`, returning a missing result for an unknown code. Preserve all original rows, columns, and indexes.

**Requirements:**
- Match currency codes exactly to the supplied fixed-rate mapping; do not fetch current exchange rates.
- Retain unknown-currency and missing-amount records with missing converted amounts.

**Available datasets and columns:**
**`transactions:`** transaction_id, customer_id, amount, currency, transaction_ts, type

**Expected output:** A Pandas DataFrame containing every original transaction column plus `amount_cad`. Each converted amount uses its currency's stated rate; unsupported codes yield missing values. Rows and indexes remain unchanged.

**Solution:**


In [ ]:
rates={'USD':1.35,'CAD':1.0,'EUR':1.48,'GBP':1.72};result=transactions.assign(amount_cad=transactions['amount']*transactions['currency'].map(rates))

display(result)


### Q76 — Customer lifetime value

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Enrich every row in `customers` with `completed_orders` and `ltv` from `orders`, matching on `customer_id`. Count orders whose `status` is `completed` and sum their amounts for lifetime value. Use zero for both measures when no completed orders match, retaining all customer fields and indexes.

**Requirements:**
- Count completed order records rather than distinct amounts or status categories.
- Preserve all customers, including those with no completed orders, without changing either input.

**Available datasets and columns:**
**`customers:`** customer_id, name, email, segment, country, signup_date
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame containing all customer columns plus `completed_orders` and `ltv`. There is one output row per customer row, with original indexes and zero for both measures when no completed orders match.

**Solution:**


In [ ]:
x=orders.loc[orders['status'].eq('completed')].groupby('customer_id').agg(completed_orders=('order_id','size'),ltv=('amount','sum'));result=customers.join(x,on='customer_id');result[['completed_orders','ltv']]=result[['completed_orders','ltv']].fillna(0)

display(result)


### Q77 — Top customers by segment

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Calculate each customer's completed-order revenue from `orders` and combine it with `customers` on `customer_id`. Include customers without completed orders at zero revenue. Within each `segment`, return the three highest-revenue customers with `customer_id`, `segment`, `name`, and revenue as `amount`, breaking equal-revenue ties with `customer_id` ascending, including at the cutoff. Treat missing segments as one separate group. Order named segments ascending, followed by the missing-segment group, and use revenue descending then customer ID ascending within each group.

**Requirements:**
- Apply the three-customer limit independently to each segment; include all available customers when a segment contains fewer than three.
- Customers with zero completed-order revenue remain eligible, and missing-segment customers must not be discarded.
- Leave `customers` and `orders` unchanged while producing the revenue ranking.

**Available datasets and columns:**
**`customers:`** customer_id, name, email, segment, country, signup_date
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A Pandas DataFrame with `customer_id`, `segment`, `name`, and `amount`, where `amount` is completed-order revenue and is zero when no completed orders match. Return up to three customers per segment, including a separate missing-segment group. Order named segments ascending, then the missing group; within each group, order `amount` descending and `customer_id` ascending.

**Solution:**


In [ ]:
x=orders.loc[orders['status'].eq('completed')].groupby('customer_id',as_index=False)['amount'].sum();y=customers[['customer_id','segment','name']].merge(x,on='customer_id',how='left').fillna({'amount':0});result=y.sort_values(['segment','amount','customer_id'],ascending=[True,False,True]).groupby('segment',dropna=False).head(3)

display(result)


### Q78 — Batch quality summary

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Return a one-row quality summary for `orders` with `rows`, `missing_customer_ids`, `duplicate_order_ids`, and `negative_amounts`. Count duplicate order IDs as occurrences after the first, not the number of distinct repeated IDs. Negative amounts are strictly below zero; zero and missing amounts should not increase that count.

**Requirements:**
- Measure each issue independently; a record may contribute to more than one metric.
- Count repeated order-ID occurrences after the first, and do not count missing or zero amounts as negative.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount

**Expected output:** A one-row Pandas DataFrame with integer `rows`, `missing_customer_ids`, `duplicate_order_ids`, and `negative_amounts`. Each field is a batch-level count, including zero where no issue is present.

**Solution:**


In [ ]:
result=pd.DataFrame([{'rows':len(orders),'missing_customer_ids':int(orders['customer_id'].isna().sum()),'duplicate_order_ids':int(orders.duplicated('order_id').sum()),'negative_amounts':int(orders['amount'].lt(0).sum())}])

display(result)


### Q79 — Pipeline daily SLA

**Interview scenario:** You are building an ETL observability system. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Summarize `pipeline_logs` by `pipeline` and the UTC calendar `day` derived from `timestamp`. Return `pipeline`, `day`, total log count as `total`, and `sla_breaches` for records with `duration_ms` strictly greater than 60,000. Retain groups with no breaches and report zero.

**Requirements:**
- Group by both pipeline and UTC calendar date; include all log levels in `total`.
- A duration of exactly 60,000 milliseconds is not a breach, and groups with no breaches remain.

**Available datasets and columns:**
**`pipeline_logs:`** log_id, pipeline, level, duration_ms, timestamp, message

**Expected output:** A Pandas DataFrame with `pipeline`, calendar-date `day`, integer `total`, and integer `sla_breaches`, one row per represented pipeline-day pair. Non-breaching groups have a breach count of zero.

**Solution:**


In [ ]:
x=pipeline_logs.assign(day=pd.to_datetime(pipeline_logs['timestamp'],utc=True).dt.date,breach=pipeline_logs['duration_ms'].gt(60000));result=x.groupby(['pipeline','day'],as_index=False).agg(total=('log_id','size'),sla_breaches=('breach','sum'))

display(result)


### Q80 — Order-to-transaction reconciliation

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using Pandas DataFrame operations.

**Task:** Compare the total `amount` of completed orders in `orders` with the total amount of purchase transactions in `transactions`. Match order status exactly to `completed` and transaction type to `purchase` without regard to case. Return one row with `completed_order_total`, `purchase_transaction_total`, and `difference` as order total minus transaction total.

**Requirements:**
- Compare independent source totals without joining individual orders to transactions.
- Report the signed difference even when it is zero; no mismatch tolerance or currency conversion is part of this exercise.

**Available datasets and columns:**
**`orders:`** order_id, customer_id, order_date, status, amount
**`transactions:`** transaction_id, customer_id, amount, currency, transaction_ts, type

**Expected output:** A one-row Pandas DataFrame with `completed_order_total`, `purchase_transaction_total`, and `difference`. The difference is the completed-order sum minus the purchase-transaction sum; totals use the original amounts.

**Solution:**


In [ ]:
order_total=orders.loc[orders['status'].eq('completed'),'amount'].sum();tx_total=transactions.loc[transactions['type'].astype('string').str.lower().eq('purchase'),'amount'].sum();result=pd.DataFrame([{'completed_order_total':order_total,'purchase_transaction_total':tx_total,'difference':order_total-tx_total}])

display(result)
